# EpidemicWatch — Phase 3: Centralized Baseline (Bio_ClinicalBERT + LoRA)

This notebook trains the centralized baseline classifier that later federated
results (Phase 4) will be compared against.

**Stack:** `emilyalsentzer/Bio_ClinicalBERT` base model, frozen, with **LoRA**
adapters (r=8) + a trainable classification head on top. Only the adapter +
head weights are trained and saved — the base model stays frozen and shared.

**Data:** pooled Symptom2Disease data from Phase 2 (1,153 records, 5 syndrome
categories, non-IID hospital shards with a pooled train/val/test split for this
centralized run).

> Run this in Google Colab with a **GPU runtime** (Runtime → Change runtime type
> → T4 GPU). CPU will work but training will be considerably slower.

## 0. Setup

In [ ]:
import os
import json
import random
import shutil
import time
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---- Config -------------------------------------------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
DATA_DIR = "epidemicwatch_data"          # Phase 2 output folder
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
MAX_EPOCHS = 8
EARLY_STOP_PATIENCE = 3
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
CLASSIFIER_DROPOUT = 0.3
OUTPUT_DIR = "phase3_baseline_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config set.")

## 1. Load Pooled Data from Phase 2

Pools the non-IID hospital shards' train/val/test splits back together, since
the centralized baseline trains on ALL data at once (federation is what splits
it apart in Phase 4). If `epidemicwatch_data/` isn't already in this Colab
session, upload the `epidemicwatch_data.zip` from Phase 2 when prompted.

In [ ]:
if not os.path.isdir(DATA_DIR):
    from google.colab import files
    print(f"'{DATA_DIR}' not found. Please upload epidemicwatch_data.zip from Phase 2.")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(DATA_DIR)
    print(f"Extracted to {DATA_DIR}/")
else:
    print(f"Found existing '{DATA_DIR}/' in this session.")

def load_pooled_split(split_name, data_dir=DATA_DIR, source="non_iid"):
    """Concatenate a given split (train/val/test) across all hospital shards."""
    hospital_dirs = sorted(
        d for d in os.listdir(os.path.join(data_dir, source))
        if d.startswith("hospital_")
    )
    dfs = []
    for h in hospital_dirs:
        path = os.path.join(data_dir, source, h, f"{split_name}.csv")
        dfs.append(pd.read_csv(path))
    return pd.concat(dfs, ignore_index=True)

train_df = load_pooled_split("train")
val_df = load_pooled_split("val")
test_df = load_pooled_split("test")

print(f"Pooled train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)}")
train_df.head()

## 2. Label Encoding

In [ ]:
CATEGORIES = sorted(train_df["syndrome_category"].unique())
label2id = {cat: i for i, cat in enumerate(CATEGORIES)}
id2label = {i: cat for cat, i in label2id.items()}
NUM_LABELS = len(CATEGORIES)

for d in (train_df, val_df, test_df):
    d["label_id"] = d["syndrome_category"].map(label2id)

print("Categories:", CATEGORIES)
print("label2id:", label2id)
print("\nClass balance in pooled train set:")
print(train_df["syndrome_category"].value_counts())

## 3. Lightweight Text Augmentation (self-contained, no external downloads)

Random word deletion + random word swap (EDA-style). Kept dependency-free since
`nlpaug`'s synonym-replacement backends need external corpora (WordNet/embeddings)
that may not be reachable in every Colab session. This still meaningfully
expands the small training set — each augmented copy is a light perturbation of
an original example, not a duplicate.

In [ ]:
def augment_text(text, p_delete=0.1, p_swap=0.1, seed=None):
    rng = random.Random(seed)
    words = text.split()
    if len(words) < 4:
        return text  # too short to safely perturb

    # random deletion
    kept = [w for w in words if rng.random() > p_delete]
    if len(kept) < 3:
        kept = words  # safety: don't over-delete short sentences

    # random adjacent swap
    if rng.random() < p_swap and len(kept) > 1:
        i = rng.randrange(len(kept) - 1)
        kept[i], kept[i + 1] = kept[i + 1], kept[i]

    return " ".join(kept)


def build_augmented_train_df(df, n_augments_per_example=1, seed=SEED):
    """Returns original + augmented copies concatenated together."""
    augmented_rows = []
    for idx, row in df.iterrows():
        for k in range(n_augments_per_example):
            new_text = augment_text(row["text"], seed=seed + idx * 7 + k)
            augmented_rows.append({**row.to_dict(), "text": new_text})
    aug_df = pd.DataFrame(augmented_rows)
    return pd.concat([df, aug_df], ignore_index=True)

# Quick sanity check
sample = train_df.iloc[0]["text"]
print("Original :", sample)
print("Augmented:", augment_text(sample, seed=1))

## 4. Tokenizer & PyTorch Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quick check for tokenization/vocabulary mismatch given the clinical-vs-casual
# domain gap noted in the project design doc.
sample_texts = train_df["text"].sample(3, random_state=SEED).tolist()
print("Sample tokenizations (watch for unusual subword splitting on everyday words):")
for t in sample_texts:
    tokens = tokenizer.tokenize(t)
    print(f"\n  Text: {t}")
    print(f"  Tokens ({len(tokens)}): {tokens}")


class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LENGTH):
        self.texts = df["text"].tolist()
        self.labels = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


val_dataset = SymptomDataset(val_df, tokenizer)
test_dataset = SymptomDataset(test_df, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Val dataset: {len(val_dataset)} | Test dataset: {len(test_dataset)}")

## 5. Model: Bio_ClinicalBERT + LoRA

In [ ]:
def build_lora_model():
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS,
        id2label=id2label, label2id=label2id,
        classifier_dropout=CLASSIFIER_DROPOUT,
    )
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=["query", "value"],   # matches Bio_ClinicalBERT's BERT-style attention layers
        modules_to_save=["classifier"],       # keep classification head trainable + saved with adapter
    )
    model = get_peft_model(base_model, lora_config)
    return model

model = build_lora_model()
model.print_trainable_parameters()
model.to(DEVICE)
print("Model ready on", DEVICE)

## 6. Training Loop (with Early Stopping)

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            total_loss += out.loss.item() * batch["labels"].size(0)
            preds = out.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted", zero_division=0
    )
    avg_loss = total_loss / len(loader.dataset)
    return {
        "loss": avg_loss, "accuracy": acc,
        "precision_macro": precision_macro, "recall_macro": recall_macro, "f1_macro": f1_macro,
        "precision_weighted": precision_w, "recall_weighted": recall_w, "f1_weighted": f1_w,
        "preds": all_preds, "labels": all_labels,
    }


def train_model(augment=False, run_name="baseline"):
    print(f"\n{'='*60}\nTraining run: {run_name} (augment={augment})\n{'='*60}")

    this_train_df = build_augmented_train_df(train_df) if augment else train_df
    train_dataset = SymptomDataset(this_train_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    print(f"Training on {len(train_dataset)} examples "
          f"({'augmented' if augment else 'original'})")

    model = build_lora_model()
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    best_f1 = -1.0
    epochs_no_improve = 0
    checkpoint_dir = os.path.join(OUTPUT_DIR, f"adapter_{run_name}")
    history = []

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
            epoch_loss += out.loss.item() * batch["labels"].size(0)
        train_loss = epoch_loss / len(train_dataset)

        val_metrics = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, **{
            k: v for k, v in val_metrics.items() if k not in ("preds", "labels")
        }})
        print(f"Epoch {epoch}: train_loss={train_loss:.4f} | "
              f"val_loss={val_metrics['loss']:.4f} | "
              f"val_acc={val_metrics['accuracy']:.4f} | "
              f"val_f1_macro={val_metrics['f1_macro']:.4f}")

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            epochs_no_improve = 0
            model.save_pretrained(checkpoint_dir)
            print(f"  -> New best val F1 ({best_f1:.4f}), checkpoint saved.")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"  -> Early stopping (no improvement for {EARLY_STOP_PATIENCE} epochs).")
                break

    training_time = time.time() - start_time
    peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if torch.cuda.is_available() else None

    # Reload best checkpoint for final test evaluation
    best_base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
    )
    best_model = PeftModel.from_pretrained(best_base, checkpoint_dir)
    best_model.to(DEVICE)
    test_metrics = evaluate(best_model, test_loader)

    adapter_size_kb = sum(
        os.path.getsize(os.path.join(checkpoint_dir, f))
        for f in os.listdir(checkpoint_dir)
    ) / 1024

    print(f"\nTraining time: {training_time:.1f}s")
    if peak_mem_mb is not None:
        print(f"Peak GPU memory: {peak_mem_mb:.1f} MB")
    print(f"Adapter checkpoint size on disk: {adapter_size_kb:.1f} KB")
    print(f"\nTest set results ({run_name}):")
    print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  F1 (macro): {test_metrics['f1_macro']:.4f} | F1 (weighted): {test_metrics['f1_weighted']:.4f}")

    return {
        "run_name": run_name, "history": history, "test_metrics": test_metrics,
        "training_time_sec": training_time, "peak_mem_mb": peak_mem_mb,
        "adapter_size_kb": adapter_size_kb, "checkpoint_dir": checkpoint_dir,
    }

## 7. Run Both Variants: With and Without Augmentation

This directly compares whether the lightweight augmentation from Section 3
actually helps on this small dataset, or just adds noise/training time for no
benefit — don't assume augmentation helps until this comparison confirms it.

In [ ]:
result_no_aug = train_model(augment=False, run_name="no_augment")
result_aug = train_model(augment=True, run_name="augmented")

## 8. Compare Results

In [ ]:
comparison_rows = []
for r in (result_no_aug, result_aug):
    tm = r["test_metrics"]
    comparison_rows.append({
        "run": r["run_name"],
        "test_accuracy": round(tm["accuracy"], 4),
        "test_f1_macro": round(tm["f1_macro"], 4),
        "test_f1_weighted": round(tm["f1_weighted"], 4),
        "training_time_sec": round(r["training_time_sec"], 1),
        "peak_gpu_mem_mb": round(r["peak_mem_mb"], 1) if r["peak_mem_mb"] else "N/A (CPU)",
        "adapter_size_kb": round(r["adapter_size_kb"], 1),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "baseline_comparison.csv"), index=False)

# Pick whichever variant did better on val F1 as "the" centralized baseline going forward
best_result = result_aug if result_aug["test_metrics"]["f1_macro"] >= result_no_aug["test_metrics"]["f1_macro"] else result_no_aug
print(f"\nBest variant for downstream comparison (Phase 4): '{best_result['run_name']}'")

## 9. Classification Report & Confusion Matrix (Best Variant)

In [ ]:
tm = best_result["test_metrics"]
print(classification_report(
    tm["labels"], tm["preds"], target_names=CATEGORIES, zero_division=0
))

cm = confusion_matrix(tm["labels"], tm["preds"])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix — Centralized Baseline ({best_result['run_name']})")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for r, label in [(result_no_aug, "no augment"), (result_aug, "augmented")]:
    hist_df = pd.DataFrame(r["history"])
    axes[0].plot(hist_df["epoch"], hist_df["val_f1_macro"], marker="o", label=label)
    axes[1].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label=f"{label} (train)")
    axes[1].plot(hist_df["epoch"], hist_df["loss"], marker="x", linestyle="--", label=f"{label} (val)")

axes[0].set_title("Validation F1 (macro) per Epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("F1 (macro)"); axes[0].legend()
axes[1].set_title("Loss per Epoch")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 11. Reloading the Saved Adapter for Inference

Demonstrates loading the base model fresh + attaching the saved LoRA adapter —
this is the pattern Phase 4 (federated) and Phase 8 (dashboard) will both reuse.

In [ ]:
def load_trained_model(checkpoint_dir):
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
    )
    model = PeftModel.from_pretrained(base, checkpoint_dir)
    model.to(DEVICE)
    model.eval()
    return model

def predict(text, model):
    enc = tokenizer(text, truncation=True, padding="max_length",
                     max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    pred_id = logits.argmax(dim=-1).item()
    return id2label[pred_id]

reloaded_model = load_trained_model(best_result["checkpoint_dir"])
sample_text = test_df.iloc[0]["text"]
print("Text:", sample_text)
print("True label:", test_df.iloc[0]["syndrome_category"])
print("Predicted:", predict(sample_text, reloaded_model))

## 12. Interpreting Your Results

**Expected F1 range for this setup:** given ~800 pooled training examples across
5 classes (some classes as thin as ~100 training examples for Respiratory) with
LoRA fine-tuning on a domain-mismatched base model (Bio_ClinicalBERT was
pretrained on formal MIMIC-III notes, not casual first-person symptom text), a
**realistic range is roughly 75-90% accuracy / macro-F1**. Symptom2Disease is a
fairly "easy" dataset in the sense that each disease has quite distinctive
symptom language, so scores at the higher end of that range are plausible —
but:

- **If you see 98%+ accuracy or macro-F1**, be suspicious. With this little
  data, that level usually means the model has memorized surface patterns
  specific to this dataset's phrasing (a known issue with Symptom2Disease,
  since it's synthetically-written text with fairly formulaic sentence
  structures per disease) rather than learning generalizable clinical
  reasoning. Check: does performance hold up on hand-written test sentences you
  type yourself (try the `predict()` function above with a symptom description
  in your own words)? A big drop there confirms overfitting to dataset-specific
  phrasing rather than the task itself.
- **If results are notably worse than this range (well under 70%)**, the most
  likely causes, in order of likelihood: (a) the domain gap between
  Bio_ClinicalBERT's clinical pretraining and this casual-text dataset is
  hurting more than expected — check the tokenization samples printed in
  Section 4 for unusual subword fragmentation on common words, which is a
  concrete symptom of this; (b) LoRA's rank (r=8) or learning rate needs
  tuning — try r=16 or a lower learning rate (5e-5); (c) check the confusion
  matrix for whether errors concentrate in the thinnest category
  (Respiratory, ~145 examples) — if so, that's expected data-scarcity
  behavior, not a bug.
- **Comparing against a hypothetical plain-DistilBERT baseline:** DistilBERT's
  general-domain pretraining is arguably closer to this dataset's casual style
  than Bio_ClinicalBERT's clinical notes, so if your Bio_ClinicalBERT numbers
  land meaningfully below ~80%, it's a legitimate, reportable finding that the
  domain-mismatch cost outweighed the benefit of clinical pretraining for this
  particular dataset — worth a paragraph in your report either way, since it's
  a real methodological insight, not just a footnote.

Record your actual numbers here once you've run this, and I can help you
interpret whether they land in the expected range.